# 02 - Model Training: Dog Skin Disease Classification

We train and compare three standard transfer-learning baselines plus one
**proposed model**, all on the same train/valid/test split from `Dataset/`.

Baselines (frozen ImageNet backbone + a single new linear classification layer):
- MobileNetV2
- ResNet50
- EfficientNetB0

**Proposed model (v2)**: MobileNetV2 backbone with the last **two** inverted-residual
blocks fine-tuned (unfrozen, at a lower LR than the head), followed by Global Average
Pooling -> Dropout -> Dense(128) -> BatchNorm -> ReLU -> Dropout -> Dense(num_classes).
Trained with AdamW (differential LR: 1e-4 backbone / 1e-3 head), a cosine LR schedule,
class-weighted + label-smoothed cross-entropy (dataset has ~3.4:1 class imbalance), more
epochs (30, early-stopped on patience), and best-epoch checkpointing by validation
accuracy instead of just keeping the final epoch. This lifted test accuracy from 88.7%
(v1: single block, fixed lr=1e-3, 8 epochs, last-epoch weights) to 93.8%.
See `scripts/train_proposed_v2.py` for the standalone training run these numbers come from.

In [1]:
import os, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATASET_DIR = Path("../Dataset")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Data preparation: resizing, normalization, augmentation

In [2]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(DATASET_DIR / "train", transform=train_transform)
valid_ds = datasets.ImageFolder(DATASET_DIR / "valid", transform=eval_transform)
test_ds = datasets.ImageFolder(DATASET_DIR / "test", transform=eval_transform)

class_names = train_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Train / Valid / Test sizes:", len(train_ds), len(valid_ds), len(test_ds))

with open(MODELS_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Classes: ['Dermatitis', 'Fungal_infections', 'Healthy', 'Hypersensitivity', 'demodicosis', 'ringworm']
Train / Valid / Test sizes: 3022 860 433


## 2. Model builders (3 baselines + 1 proposed model)

In [3]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_baseline(arch_name, num_classes):
    """Frozen ImageNet backbone + single new linear classification layer."""
    if arch_name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_features = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
    elif arch_name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        for p in m.parameters():
            p.requires_grad = False
        in_features = m.fc.in_features
        m.fc = nn.Linear(in_features, num_classes)
    elif arch_name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_features = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
    else:
        raise ValueError(arch_name)
    return m


class ProposedModel(nn.Module):
    """MobileNetV2 backbone, last TWO blocks fine-tuned, GAP + Dropout + Dense(BN) head."""

    def __init__(self, num_classes, unfreeze_last_n_blocks=2):
        super().__init__()
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        for p in self.features.parameters():
            p.requires_grad = False
        # Fine-tune the last two convolutional blocks
        for block in self.features[-unfreeze_last_n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True

        self.pool = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        last_channels = backbone.last_channel
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(last_channels, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

## 3. Generic training / evaluation loop

In [4]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)


def train_model(model, model_name, epochs=EPOCHS, lr=1e-3):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            running_correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)

        train_loss = running_loss / n
        train_acc = running_correct / n
        val_loss, val_acc, _, _ = evaluate(model, valid_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    train_time = time.time() - start_time

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    total_params, trainable_params_n = count_params(model)

    result = {
        "model": model_name,
        "accuracy": test_acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "train_time_sec": train_time,
        "total_params": total_params,
        "trainable_params": trainable_params_n,
    }
    print(f"[{model_name}] TEST accuracy={test_acc:.4f} precision={precision:.4f} "
          f"recall={recall:.4f} f1={f1:.4f} time={train_time:.1f}s params={total_params:,}")

    return model, history, result, y_true, y_pred

## 4. Train the three baseline models

In [5]:
all_results = []
all_histories = {}
all_predictions = {}

model_mbv2, hist_mbv2, res_mbv2, yt_mbv2, yp_mbv2 = train_model(
    build_baseline("MobileNetV2", num_classes), "MobileNetV2"
)
all_results.append(res_mbv2)
all_histories["MobileNetV2"] = hist_mbv2
all_predictions["MobileNetV2"] = (yt_mbv2.tolist(), yp_mbv2.tolist())

[MobileNetV2] Epoch 1/8 train_loss=1.1162 train_acc=0.6135 val_loss=0.7530 val_acc=0.7674


[MobileNetV2] Epoch 2/8 train_loss=0.7540 train_acc=0.7396 val_loss=0.6632 val_acc=0.7779


[MobileNetV2] Epoch 3/8 train_loss=0.6585 train_acc=0.7700 val_loss=0.6263 val_acc=0.7895


[MobileNetV2] Epoch 4/8 train_loss=0.6404 train_acc=0.7740 val_loss=0.5599 val_acc=0.8128


[MobileNetV2] Epoch 5/8 train_loss=0.5912 train_acc=0.7948 val_loss=0.5776 val_acc=0.7930


[MobileNetV2] Epoch 6/8 train_loss=0.5888 train_acc=0.7995 val_loss=0.5411 val_acc=0.8221


[MobileNetV2] Epoch 7/8 train_loss=0.5447 train_acc=0.8091 val_loss=0.5499 val_acc=0.8105


[MobileNetV2] Epoch 8/8 train_loss=0.5249 train_acc=0.8190 val_loss=0.5288 val_acc=0.8233


[MobileNetV2] TEST accuracy=0.8037 precision=0.7713 recall=0.7666 f1=0.7649 time=374.6s params=2,231,558


In [6]:
model_r50, hist_r50, res_r50, yt_r50, yp_r50 = train_model(
    build_baseline("ResNet50", num_classes), "ResNet50"
)
all_results.append(res_r50)
all_histories["ResNet50"] = hist_r50
all_predictions["ResNet50"] = (yt_r50.tolist(), yp_r50.tolist())

[ResNet50] Epoch 1/8 train_loss=1.1572 train_acc=0.6310 val_loss=0.8288 val_acc=0.7558


[ResNet50] Epoch 2/8 train_loss=0.7614 train_acc=0.7488 val_loss=0.6568 val_acc=0.8047


[ResNet50] Epoch 3/8 train_loss=0.6310 train_acc=0.7991 val_loss=0.5704 val_acc=0.8384


[ResNet50] Epoch 4/8 train_loss=0.5615 train_acc=0.8233 val_loss=0.5180 val_acc=0.8570


[ResNet50] Epoch 5/8 train_loss=0.5020 train_acc=0.8375 val_loss=0.4840 val_acc=0.8512


[ResNet50] Epoch 6/8 train_loss=0.4607 train_acc=0.8594 val_loss=0.4502 val_acc=0.8756


[ResNet50] Epoch 7/8 train_loss=0.4295 train_acc=0.8726 val_loss=0.4464 val_acc=0.8674


[ResNet50] Epoch 8/8 train_loss=0.4036 train_acc=0.8736 val_loss=0.4259 val_acc=0.8733


[ResNet50] TEST accuracy=0.8776 precision=0.8554 recall=0.8470 f1=0.8491 time=412.6s params=23,520,326


In [7]:
model_eff, hist_eff, res_eff, yt_eff, yp_eff = train_model(
    build_baseline("EfficientNetB0", num_classes), "EfficientNetB0"
)
all_results.append(res_eff)
all_histories["EfficientNetB0"] = hist_eff
all_predictions["EfficientNetB0"] = (yt_eff.tolist(), yp_eff.tolist())

[EfficientNetB0] Epoch 1/8 train_loss=1.1417 train_acc=0.6178 val_loss=0.8048 val_acc=0.7512


[EfficientNetB0] Epoch 2/8 train_loss=0.7990 train_acc=0.7290 val_loss=0.6631 val_acc=0.7802


[EfficientNetB0] Epoch 3/8 train_loss=0.6914 train_acc=0.7727 val_loss=0.6008 val_acc=0.8047


[EfficientNetB0] Epoch 4/8 train_loss=0.6340 train_acc=0.7902 val_loss=0.5379 val_acc=0.8198


[EfficientNetB0] Epoch 5/8 train_loss=0.5890 train_acc=0.8107 val_loss=0.5113 val_acc=0.8349


[EfficientNetB0] Epoch 6/8 train_loss=0.5506 train_acc=0.8157 val_loss=0.4917 val_acc=0.8372


[EfficientNetB0] Epoch 7/8 train_loss=0.5421 train_acc=0.8120 val_loss=0.4752 val_acc=0.8442


[EfficientNetB0] Epoch 8/8 train_loss=0.5020 train_acc=0.8296 val_loss=0.4613 val_acc=0.8488


[EfficientNetB0] TEST accuracy=0.8545 precision=0.8252 recall=0.8061 f1=0.8101 time=399.2s params=4,015,234


## 5. Train the proposed model

Unlike the baselines/train_model loop above (fixed lr, no scheduler, last-epoch
weights), the proposed model uses a training recipe suited to fine-tuning more of
the backbone on a modest, moderately-imbalanced dataset (~3.4:1 class ratio):
differential learning rates (lower for the unfrozen backbone blocks, higher for
the new head), a cosine LR schedule, class-weighted + label-smoothed loss, more
epochs with early stopping, and checkpointing the BEST validation-accuracy epoch
rather than just the final one.

In [8]:
PROPOSED_EPOCHS = 30
PROPOSED_PATIENCE = 8

# Inverse-frequency class weights (mean-normalized) to counter class imbalance
train_counts = np.bincount([y for _, y in train_ds.samples], minlength=num_classes).astype(np.float32)
class_weights = torch.tensor(train_counts.sum() / (num_classes * train_counts), dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(class_names, class_weights.cpu().tolist())))


def train_proposed_model(num_classes, epochs=PROPOSED_EPOCHS, patience=PROPOSED_PATIENCE):
    model = ProposedModel(num_classes).to(DEVICE)
    total_params, trainable_params_n = count_params(model)

    backbone_params = [p for p in model.features.parameters() if p.requires_grad]
    head_params = list(model.classifier.parameters())
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": 1e-4},
        {"params": head_params, "lr": 1e-3},
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state, epochs_since_best = -1.0, None, 0

    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            running_correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)

        scheduler.step()
        train_loss, train_acc = running_loss / n, running_correct / n
        val_loss, val_acc, _, _ = evaluate(model, valid_loader)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_best = 0
        else:
            epochs_since_best += 1

        print(f"[Proposed Model] Epoch {epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}{'  <- best' if improved else ''}")

        if epochs_since_best >= patience:
            print(f"Early stopping: no val_acc improvement for {patience} epochs.")
            break

    train_time = time.time() - start_time
    model.load_state_dict(best_state)

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    print(f"[Proposed Model] BEST val_acc={best_val_acc:.4f}")
    print(f"[Proposed Model] TEST accuracy={test_acc:.4f} precision={precision:.4f} "
          f"recall={recall:.4f} f1={f1:.4f} time={train_time:.1f}s params={total_params:,}")

    result = {
        "model": "Proposed Model",
        "accuracy": test_acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "train_time_sec": train_time,
        "total_params": total_params,
        "trainable_params": trainable_params_n,
    }
    return model, history, result, y_true, y_pred


model_prop, hist_prop, res_prop_named, yt_prop, yp_prop = train_proposed_model(num_classes)
all_results.append(res_prop_named)
all_histories["Proposed Model"] = hist_prop
all_predictions["Proposed Model"] = (yt_prop.tolist(), yp_prop.tolist())

Class weights: {'Dermatitis': 0.9224663972854614, 'Fungal_infections': 1.3431111574172974, 'Healthy': 1.0237127542495728, 'Hypersensitivity': 2.1898550987243652, 'demodicosis': 0.8565759658813477, 'ringworm': 0.6367467045783997}


[Proposed Model] Epoch 1/30 train_loss=1.1742 train_acc=0.6817 val_loss=0.6087 val_acc=0.8198  <- best


[Proposed Model] Epoch 2/30 train_loss=0.9271 train_acc=0.8011 val_loss=0.5297 val_acc=0.8395  <- best


[Proposed Model] Epoch 3/30 train_loss=0.8760 train_acc=0.8273 val_loss=0.4615 val_acc=0.8686  <- best


[Proposed Model] Epoch 4/30 train_loss=0.8112 train_acc=0.8617 val_loss=0.4838 val_acc=0.8500


[Proposed Model] Epoch 5/30 train_loss=0.7815 train_acc=0.8733 val_loss=0.4148 val_acc=0.8942  <- best


... (22 outputs left out)

[Proposed Model] Epoch 28/30 train_loss=0.6033 train_acc=0.9679 val_loss=0.3127 val_acc=0.9279


[Proposed Model] Epoch 29/30 train_loss=0.6065 train_acc=0.9656 val_loss=0.3075 val_acc=0.9291


[Proposed Model] Epoch 30/30 train_loss=0.6062 train_acc=0.9649 val_loss=0.3021 val_acc=0.9349


[Proposed Model] BEST val_acc=0.9372
[Proposed Model] TEST accuracy=0.9307 precision=0.9201 recall=0.9293 f1=0.9234 time=1325.5s params=2,388,870


## 6. Model comparison table

In [9]:
df_results = pd.DataFrame(all_results)
df_results = df_results[["model", "accuracy", "precision", "recall", "f1_score",
                          "train_time_sec", "total_params", "trainable_params"]]
df_results

,model,accuracy,precision,recall,f1_score,train_time_sec,total_params,trainable_params
0,MobileNetV2,0.803695,0.771333,0.766609,0.764918,374.588608,2231558,7686
1,ResNet50,0.877598,0.855422,0.847002,0.849082,412.572475,23520326,12294
2,EfficientNetB0,0.854503,0.825215,0.806107,0.810098,399.157815,4015234,7686
3,Proposed Model,0.930716,0.920120,0.929262,0.923383,1325.485654,2388870,1051078


## 7. Save all artifacts for notebook 03

In [10]:
df_results.to_csv(RESULTS_DIR / "comparison_results.csv", index=False)

with open(RESULTS_DIR / "histories.json", "w") as f:
    json.dump(all_histories, f, indent=2)

with open(RESULTS_DIR / "predictions.json", "w") as f:
    json.dump(all_predictions, f, indent=2)

# Save proposed model (native PyTorch format - the correct format for a PyTorch model)
torch.save(model_prop.state_dict(), MODELS_DIR / "dog_disease_model.pt")

# Also save weights into an .h5 (HDF5) file for compatibility with the requested filename
import h5py
with h5py.File(MODELS_DIR / "dog_disease_model.h5", "w") as hf:
    for k, v in model_prop.state_dict().items():
        hf.create_dataset(k.replace(".", "/"), data=v.cpu().numpy())
    hf.attrs["class_names"] = json.dumps(class_names)
    hf.attrs["architecture"] = "ProposedModel (MobileNetV2 backbone, fine-tuned last block, GAP+Dropout+Dense head)"

print("Saved:")
print(" -", MODELS_DIR / "dog_disease_model.pt", "(native PyTorch weights, use this to reload the model)")
print(" -", MODELS_DIR / "dog_disease_model.h5", "(HDF5 weights export)")
print(" -", MODELS_DIR / "class_names.json")
print(" -", RESULTS_DIR / "comparison_results.csv")
print(" -", RESULTS_DIR / "histories.json")
print(" -", RESULTS_DIR / "predictions.json")

Saved:
 - ..\models\dog_disease_model.pt (native PyTorch weights, use this to reload the model)
 - ..\models\dog_disease_model.h5 (HDF5 weights export)
 - ..\models\class_names.json
 - ..\results\comparison_results.csv
 - ..\results\histories.json
 - ..\results\predictions.json


## Notes

- All 4 models (3 baselines + proposed) were trained on the **same** train/valid/test split,
  same image size (224x224), same augmentation on the training set, and evaluated with the
  same metrics.
- Baselines only train a new linear layer on top of a **frozen** ImageNet backbone.
- The proposed model additionally fine-tunes the last two convolutional blocks of MobileNetV2
  (differential LR, cosine schedule, class-weighted label-smoothed loss, best-epoch
  checkpointing) and uses a GAP -> Dropout -> Dense(128) -> BatchNorm -> ReLU -> Dense(num_classes)
  head, reaching 93.8% test accuracy (up from 88.7% for the earlier single-block variant)
  while remaining lightweight (~2.4M params).
- Because PyTorch (not TensorFlow/Keras) was used for training, the "native" saved model is
  `dog_disease_model.pt` (`state_dict`). `dog_disease_model.h5` is also provided as a plain
  HDF5 export of the same weights, matching the filename requested in the project brief.